# AI-Task8: Research Report Generator (LCEL)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jaweriamushtaq41-coder/Smart_District_2/blob/main/AI-Task8/AI_Task8_Research_Report_Generator.ipynb)

**Kya banana hai:** Raw topic -> key-facts aur counterarguments prompts `RunnableParallel` se ek saath -> `RunnableLambda` se jodna -> final prompt jo balanced report likhta hai -> `.txt` file.

**Concepts:** LCEL `|`, RunnableParallel, RunnableLambda, prompt chaining

**Colab mein kaise chalayen:**
1. Upar *Open In Colab* dabao (ya ye file Colab mein upload karo).
2. *Runtime > Run all* dabao.
3. Keys ho to Colab ke left side ke **Secrets** (chaabi ka icon) mein daalo. Bina key ke bhi chalta hai (**demo mode**, nakli jawab).

> Notebook ko share karne se pehle apni keys kabhi output / cell mein mat chhodo.

## 1. Libraries install aur keys (`.env`)

In [1]:
!pip -q install -U python-dotenv langchain-core langchain-openai langchain-anthropic langchain-google-genai langchain-groq

In [2]:
import os

# .env file banao. Keys do tareeqon se aa sakti hain:
#  1) Colab "Secrets" (left side ka chaabi wala icon), secret ka naam wahi jo neeche list mein hai, "Notebook access" ON
#  2) apni .env file Colab ke "Files" panel (left side ka folder icon) mein drag-drop kar do, run karne se pehle
NAMES = ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GOOGLE_API_KEY", "GROQ_API_KEY"]

existing = {}                       # pehle se .env ho (upload ki hui) to uski keys bacha lo
if os.path.exists(".env"):
    with open(".env") as f:
        for line in f:
            if "=" in line and not line.startswith("#"):
                key, value = line.strip().split("=", 1)
                existing[key] = value

lines = []
for name in NAMES:
    value = existing.get(name, "")
    try:
        from google.colab import userdata
        value = userdata.get(name) or value
    except Exception:          # Colab nahi hai, ya ye secret nahi banaya
        pass
    lines.append(f"{name}={value}")

with open(".env", "w") as f:
    f.write("\n".join(lines) + "\n")

from dotenv import load_dotenv
load_dotenv(".env", override=True)

found = [n for n in NAMES if os.getenv(n)]
print("Keys mili:", found if found else "koi nahi (demo mode chalega)")

Keys mili: koi nahi (demo mode chalega)


## 2. Model factory
`get_chat_model("groq")` jaisa call asli LangChain model deta hai (ChatOpenAI, ChatAnthropic, ChatGoogleGenerativeAI, ChatGroq).
Provider badalna ho to sirf naam badlo, chain wahi rehti hai. `demo_model` / `demo_stream_model` key ke baghair nakli model dete hain jo chain mein asli ki tarah lagte hain.

In [3]:
import os

from dotenv import load_dotenv
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from langchain_core.messages import AIMessage
from langchain_core.runnables import RunnableLambda

load_dotenv()

PROVIDERS = ["openai", "claude", "gemini", "groq"]

KEY_NAMES = {
    "openai": "OPENAI_API_KEY",
    "claude": "ANTHROPIC_API_KEY",
    "gemini": "GOOGLE_API_KEY",
    "groq": "GROQ_API_KEY",
}

MODELS = {
    "openai": os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
    "claude": os.getenv("ANTHROPIC_MODEL", "claude-3-5-haiku-latest"),
    "gemini": os.getenv("GEMINI_MODEL", "gemini-1.5-flash"),
    "groq": os.getenv("GROQ_MODEL", "llama-3.1-8b-instant"),
}


def available_providers():
    """Jin providers ki key .env mein hai."""
    return [p for p in PROVIDERS if os.getenv(KEY_NAMES[p], "").strip()]


def get_chat_model(provider, temperature=0.3):
    """Asli LangChain model wapas deta hai. Import andar hai taake sirf wahi package chahiye jo use ho."""
    if provider == "openai":
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=MODELS["openai"], temperature=temperature)
    if provider == "claude":
        from langchain_anthropic import ChatAnthropic
        return ChatAnthropic(model=MODELS["claude"], temperature=temperature)
    if provider == "gemini":
        from langchain_google_genai import ChatGoogleGenerativeAI
        return ChatGoogleGenerativeAI(model=MODELS["gemini"], temperature=temperature)
    if provider == "groq":
        from langchain_groq import ChatGroq
        return ChatGroq(model=MODELS["groq"], temperature=temperature)
    raise ValueError(f"Unknown provider: {provider}")


# ---------------- demo (nakli) models ----------------
def demo_stream_model(text, delay=0.004):
    """Ek fixed jawab deta hai, harf harf stream hota hai (asli streaming jaisa)."""
    return FakeListChatModel(responses=[text], sleep=delay)


def demo_model(responder):
    """responder(prompt_text) -> jawab ka text. Prompt dekh kar jawab chunta hai,
    is liye RunnableParallel mein (jahan tarteeb pakki nahi) bhi theek chalta hai."""
    def run(prompt_value):
        text = prompt_value if isinstance(prompt_value, str) else prompt_value.to_string()
        return AIMessage(content=responder(text))
    return RunnableLambda(run)

## 3. Chain ke hisse
```
raw topic -> dict -> [key facts || counterarguments || topic] (parallel) -> merge (lambda) -> report prompt -> model -> text
```

In [4]:
import os
import re
from datetime import datetime
from operator import itemgetter

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableParallel

In [5]:
facts_prompt = ChatPromptTemplate.from_template(
    "List 5 key facts about this topic as short bullet points. Topic: {topic}")

counter_prompt = ChatPromptTemplate.from_template(
    "List 4 strong counterarguments or criticisms about this topic as short bullet points. Topic: {topic}")

report_prompt = ChatPromptTemplate.from_template(
    "Write a balanced research report on the topic below, using ONLY the material given.\n"
    "Structure: Title, Introduction, Key Findings, Counterarguments, Conclusion.\n\n"
    "Topic: {topic}\n\n{combined}")


def merge_outputs(data):
    """Dono parallel jawab ek text mein jodta hai (RunnableLambda ka kaam)."""
    combined = (f"KEY FACTS:\n{data['key_facts']}\n\n"
                f"COUNTERARGUMENTS:\n{data['counterarguments']}")
    return {"topic": data["topic"], "combined": combined}


def build_chain(llm):
    to_dict = RunnableLambda(lambda raw_topic: {"topic": raw_topic.strip()})       # raw text -> dict

    parallel = RunnableParallel(                                                    # dono ek saath chalte hain
        key_facts=facts_prompt | llm | StrOutputParser(),
        counterarguments=counter_prompt | llm | StrOutputParser(),
        topic=RunnableLambda(itemgetter("topic")),                                  # topic aage bhejna
    )
    merge = RunnableLambda(merge_outputs)
    writer = report_prompt | llm | StrOutputParser()

    return to_dict | parallel | merge | writer


# ---------------- demo (nakli model) ----------------
def demo_brain(prompt_text):
    topic = re.search(r"Topic: (.+)", prompt_text).group(1).split("\n")[0].strip()
    if "key facts" in prompt_text.lower() and "List 5" in prompt_text:
        return (f"- {topic} has grown quickly in the last few years.\n- Many users report lower running costs.\n"
                "- Government policy strongly affects adoption.\n- Infrastructure is still developing.\n"
                "- Younger people adopt it faster.")
    if "List 4" in prompt_text:
        return ("- Upfront cost is high for most families.\n- Benefits depend on reliable infrastructure.\n"
                "- Some claimed savings are overestimated.\n- Long-term effects are not yet fully known.")
    facts = prompt_text.split("KEY FACTS:")[1].split("COUNTERARGUMENTS:")[0].strip()
    counter = prompt_text.split("COUNTERARGUMENTS:")[1].strip()
    return (f"TITLE: A Balanced Look at {topic}\n\nINTRODUCTION\n{topic} is a topic with real promise and real limits.\n\n"
            f"KEY FINDINGS\n{facts}\n\nCOUNTERARGUMENTS\n{counter}\n\n"
            f"CONCLUSION\nThe evidence supports {topic} in the right conditions, but costs and gaps must be addressed first.")

## 4. Provider chuno

In [6]:
FORCE_DEMO = False      # True likhoge to hamesha nakli model
PROVIDER = None         # "groq", "openai", "claude" ya "gemini". None = jis ki key mili
demo = FORCE_DEMO
provider = PROVIDER or (available_providers() or [None])[0]
if provider is None:
    print("[!] Koi API key nahi mili, DEMO mode chal raha hai.")
    demo, provider = True, "groq"
print("provider =", provider, "| demo =", demo)

[!] Koi API key nahi mili, DEMO mode chal raha hai.
provider = groq | demo = True


## 5. Chalao
Sirf `TOPIC` badlo.

In [7]:
TOPIC = "Electric cars in Pakistan"

llm = demo_model(demo_brain) if demo else get_chat_model(provider, temperature=0.4)
chain = build_chain(llm)
report = chain.invoke(TOPIC)

os.makedirs("reports", exist_ok=True)
safe_name = re.sub(r"[^a-z0-9]+", "_", TOPIC.lower()).strip("_")[:40]
path = os.path.join("reports", f"report_{safe_name}_{datetime.now():%Y%m%d_%H%M%S}.txt")
with open(path, "w", encoding="utf-8") as f:
    f.write(report)

print(report)
print("\nReport save ho gayi:", path)

TITLE: A Balanced Look at Electric cars in Pakistan

INTRODUCTION
Electric cars in Pakistan is a topic with real promise and real limits.

KEY FINDINGS
- Electric cars in Pakistan has grown quickly in the last few years.
- Many users report lower running costs.
- Government policy strongly affects adoption.
- Infrastructure is still developing.
- Younger people adopt it faster.

COUNTERARGUMENTS
- Upfront cost is high for most families.
- Benefits depend on reliable infrastructure.
- Some claimed savings are overestimated.
- Long-term effects are not yet fully known.

CONCLUSION
The evidence supports Electric cars in Pakistan in the right conditions, but costs and gaps must be addressed first.

Report save ho gayi: reports/report_electric_cars_in_pakistan_20261031_180746.txt


## Kya seekha
- `RunnableParallel` do independent kaam ek saath chalata hai (waqt bachta hai).
- `RunnableLambda` apna Python function chain ka hissa bana deta hai.
- `|` se prompt, model aur parser ek chain ban jate hain.